# ClaimWise Insurance Prediction — 02. Data Preprocessing

## 1. Objective

This notebook runs the **single authoritative preprocessing pipeline** of the project and then
verifies every stage it produced.

The project has exactly **one** preprocessing implementation:

`Src/claimwise_preprocessing_pipeline.py`

Nothing is copied into this notebook — the script itself is executed, so the notebook and the
command-line pipeline can never drift apart.

```text
Dataset/01_raw_claimwise_50000.csv            (50,000 x 132, includes id + loss)
        |  cleaning, duplicate removal,
        |  median/mode imputation, drop 'id'
Dataset/02_cleaned_dedup_median_imputed.csv   (50,000 x 131)
        |  label encoding of categorical columns
Dataset/03_encoded_label.csv                  (50,000 x 131, all numeric)
        |  StandardScaler on cont1..cont14 (fit on the 80% training rows only)
Dataset/04_scaled_standardized.csv            (50,000 x 131)
        |  declare final dataset
Dataset/05_preprocessed_final.csv             (50,000 x 131)
        |  train_test_split(test_size=0.20, random_state=42)
Dataset/06_split_X_train.csv   (40,000 x 130)
Dataset/06_split_X_test.csv    (10,000 x 130)
Dataset/06_split_y_train.csv   (40,000 x 1)
Dataset/06_split_y_test.csv    (10,000 x 1)
```

**Data-leakage fix applied to the pipeline:** the `StandardScaler` is fitted **only on the
training rows** (the same 80% rows selected by `train_test_split(..., random_state=42)`) and only
then applied to the whole dataset. Earlier versions fitted the scaler on all 50,000 rows, which
let test-set statistics leak into the training data.

## 2. Import Libraries

In [1]:
import runpy

import pandas as pd

In [2]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Walk upwards from the notebook folder until the ClaimWise project is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "Src").is_dir():
            return candidate
    raise FileNotFoundError(f"ClaimWise project root not found above: {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "Dataset"
SRC_DIR = PROJECT_ROOT / "Src"
MODELS_DIR = PROJECT_ROOT / "Models"
OUTPUTS_DIR = PROJECT_ROOT / "Outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root  :", PROJECT_ROOT)
print("Dataset folder:", DATASET_DIR)
print("Src folder    :", SRC_DIR)

Project root  : /Users/padaltiruvinayak/Desktop/Credit ML
Dataset folder: /Users/padaltiruvinayak/Desktop/Credit ML/Dataset
Src folder    : /Users/padaltiruvinayak/Desktop/Credit ML/Src


## 3. Run the Authoritative Preprocessing Pipeline

Below the project script is executed exactly as it runs from the command line. Its printed output
shows every stage (load -> clean/impute -> encode -> scale -> final dataset + split).

In [3]:
# Execute the single authoritative preprocessing script (no logic is duplicated here).
pipeline_namespace = runpy.run_path(str(SRC_DIR / "claimwise_preprocessing_pipeline.py"))
print("Pipeline script executed:", pipeline_namespace["__file__"])


1. STEP 1: LOAD DATASET AND AUDIT (01_raw_claimwise_50000.csv)


Original Dataset Shape: (50000, 132)

2. STEP 2: CLEANING & IMPUTATION -> 02_cleaned_dedup_median_imputed.csv


Removed column: 'id' (Unique identifier with no predictive value).


Saved Step 2 dataset: /Users/padaltiruvinayak/Desktop/Credit ML/Dataset/02_cleaned_dedup_median_imputed.csv

3. STEP 3: CATEGORICAL LABEL ENCODING -> 03_encoded_label.csv


Saved Step 3 dataset: /Users/padaltiruvinayak/Desktop/Credit ML/Dataset/03_encoded_label.csv

4. STEP 4: FEATURE SCALING -> 04_scaled_standardized.csv


Saved Step 4 dataset: /Users/padaltiruvinayak/Desktop/Credit ML/Dataset/04_scaled_standardized.csv

5. STEP 5: FINAL DATASET & SPLITTING -> 05_preprocessed_final.csv


Saved Preprocessed Dataset: /Users/padaltiruvinayak/Desktop/Credit ML/Dataset/05_preprocessed_final.csv
Saved X_train              : /Users/padaltiruvinayak/Desktop/Credit ML/Dataset/06_split_X_train.csv
Saved X_test               : /Users/padaltiruvinayak/Desktop/Credit ML/Dataset/06_split_X_test.csv
Saved y_train              : /Users/padaltiruvinayak/Desktop/Credit ML/Dataset/06_split_y_train.csv
Saved y_test               : /Users/padaltiruvinayak/Desktop/Credit ML/Dataset/06_split_y_test.csv

========== FINAL PREPROCESSING SUMMARY ==========
Original dataset rows  : 50000
Final dataset rows     : 50000
Original columns       : 132
Final columns          : 131
Training dataset rows  : 40000
Testing dataset rows   : 10000
Missing values remaining: 0
Duplicate rows remaining: 0
-------------------------------------------------
Target Column          : loss
Removed Columns        : ['id']
Total Input Features   : 130
ClaimWise Insurance Prediction preprocessing completed successfully.

## 4. Verify Every Preprocessing Stage

Each intermediate file is re-read from disk and checked for its shape, missing values and
duplicates. The table below is generated from the actual files.

> Note: for the two `y_*` files (a single `loss` column) the "duplicate rows" column only counts
> repeated loss *values*, not duplicated records. The feature files and the full datasets are the
> ones where duplicate rows matter.

In [4]:
stage_files = [
    ("1. Raw", "01_raw_claimwise_50000.csv"),
    ("2. Cleaned + imputed", "02_cleaned_dedup_median_imputed.csv"),
    ("3. Label encoded", "03_encoded_label.csv"),
    ("4. Scaled", "04_scaled_standardized.csv"),
    ("5. Final preprocessed", "05_preprocessed_final.csv"),
    ("6a. X_train", "06_split_X_train.csv"),
    ("6b. X_test", "06_split_X_test.csv"),
    ("6c. y_train", "06_split_y_train.csv"),
    ("6d. y_test", "06_split_y_test.csv"),
]

stage_rows = []
for label, filename in stage_files:
    path = DATASET_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Expected preprocessing output is missing: {path}")
    frame = pd.read_csv(path)
    stage_rows.append({
        "stage": label,
        "file": filename,
        "rows": frame.shape[0],
        "columns": frame.shape[1],
        "missing_values": int(frame.isnull().sum().sum()),
        "duplicate_rows": int(frame.duplicated().sum()),
    })

stage_summary = pd.DataFrame(stage_rows)
display(stage_summary)

PREPROCESSING_DIR = OUTPUTS_DIR / "Preprocessing"
PREPROCESSING_DIR.mkdir(parents=True, exist_ok=True)
stage_path = PREPROCESSING_DIR / "claimwise_preprocessing_stage_summary.csv"
stage_summary.to_csv(stage_path, index=False)
print("Saved:", stage_path)

,stage,file,rows,columns,missing_values,duplicate_rows
0,1. Raw,01_raw_claimwise_50000.csv,50000,132,0,0
1,2. Cleaned + imputed,02_cleaned_dedup_median_imputed.csv,50000,131,0,0
2,3. Label encoded,03_encoded_label.csv,50000,131,0,0
3,4. Scaled,04_scaled_standardized.csv,50000,131,0,0
4,5. Final preprocessed,05_preprocessed_final.csv,50000,131,0,0
5,6a. X_train,06_split_X_train.csv,40000,130,0,0
6,6b. X_test,06_split_X_test.csv,10000,130,0,0
7,6c. y_train,06_split_y_train.csv,40000,1,0,1554
8,6d. y_test,06_split_y_test.csv,10000,1,0,129


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/Preprocessing/claimwise_preprocessing_stage_summary.csv


## 5. Train / Test Data

The four model-ready files are loaded and the feature/target matrices are checked:

- `X` must contain the input features only — the target `loss` may **never** appear inside it.
- Features and targets must line up row by row.
- Everything must be numeric and free of missing values.

In [5]:
X_train = pd.read_csv(DATASET_DIR / "06_split_X_train.csv")
X_test = pd.read_csv(DATASET_DIR / "06_split_X_test.csv")
y_train = pd.read_csv(DATASET_DIR / "06_split_y_train.csv")["loss"]
y_test = pd.read_csv(DATASET_DIR / "06_split_y_test.csv")["loss"]
final = pd.read_csv(DATASET_DIR / "05_preprocessed_final.csv")

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
print("Input features:", X_train.shape[1])
print("Target         :", y_train.name)

X_train: (40000, 130) | X_test: (10000, 130)
y_train: (40000,) | y_test: (10000,)
Input features: 130
Target         : loss


## 6. Data Leakage Checks

These checks must all pass before any model is trained.

In [6]:
continuous_features = [col for col in X_train.columns if col.startswith("cont")]

checks = {
    "target 'loss' is NOT a column of X_train": "loss" not in X_train.columns,
    "target 'loss' is NOT a column of X_test": "loss" not in X_test.columns,
    "all input features are numeric": all(
        pd.api.types.is_numeric_dtype(dtype) for dtype in X_train.dtypes
    ),
    "X_train rows match y_train rows": len(X_train) == len(y_train),
    "X_test rows match y_test rows": len(X_test) == len(y_test),
    "X_train and X_test have identical columns": list(X_train.columns) == list(X_test.columns),
    "X columns == final dataset columns minus target": list(X_train.columns)
    == [col for col in final.columns if col != "loss"],
    "no missing values in the features": not (
        X_train.isna().any().any() or X_test.isna().any().any()
    ),
    "no missing values in the target": not (y_train.isna().any() or y_test.isna().any()),
    "80/20 row split (40000 / 10000)": len(X_train) == 40000 and len(X_test) == 10000,
}

checks_report = pd.DataFrame(
    [{"check": name, "result": "PASS" if ok else "FAIL"} for name, ok in checks.items()]
)
display(checks_report)

failed = [name for name, ok in checks.items() if not ok]
if failed:
    raise AssertionError(f"Data leakage checks failed: {failed}")
print("All", len(checks), "data-leakage checks passed.")

,check,result
0,target 'loss' is NOT a column of X_train,PASS
1,target 'loss' is NOT a column of X_test,PASS
2,all input features are numeric,PASS
3,X_train rows match y_train rows,PASS
4,X_test rows match y_test rows,PASS
5,X_train and X_test have identical columns,PASS
6,X columns == final dataset columns minus target,PASS
7,no missing values in the features,PASS
8,no missing values in the target,PASS
9,80/20 row split (40000 / 10000),PASS


All 10 data-leakage checks passed.


### Evidence that the scaler was fitted on training rows only

If the scaler had been fitted on all 50,000 rows, both splits would show a mean of ~0. Because it
is fitted on the 40,000 training rows, only `X_train` is centred at 0 — `X_test` keeps its own
slightly shifted distribution, which is exactly what we want.

In [7]:
train_max_abs_mean = float(X_train[continuous_features].mean().abs().max())
test_max_abs_mean = float(X_test[continuous_features].mean().abs().max())

print("Max |mean| of scaled features in X_train :", round(train_max_abs_mean, 6),
      "-> ~0, the scaler was fitted on these rows")
print("Max |mean| of scaled features in X_test  :", round(test_max_abs_mean, 6),
      "-> not 0, proving test rows were not used to fit the scaler")
print("X_train scaled-feature std min/max      :",
      round(float(X_train[continuous_features].std().min()), 6),
      round(float(X_train[continuous_features].std().max()), 6))

Max |mean| of scaled features in X_train : 0.0 -> ~0, the scaler was fitted on these rows
Max |mean| of scaled features in X_test  : 0.023239 -> not 0, proving test rows were not used to fit the scaler
X_train scaled-feature std min/max      : 1.000013 1.000013


## 7. Stage-by-Stage Consistency

Confirm that the stages really are the expected transformations of each other (identifier removed,
continuous features scaled, target untouched).

In [8]:
raw = pd.read_csv(DATASET_DIR / "01_raw_claimwise_50000.csv")
clean = pd.read_csv(DATASET_DIR / "02_cleaned_dedup_median_imputed.csv")
encoded = pd.read_csv(DATASET_DIR / "03_encoded_label.csv")
scaled = pd.read_csv(DATASET_DIR / "04_scaled_standardized.csv")

print("Raw        :", raw.shape, "| 'id' present:", "id" in raw.columns, "| 'loss' present:", "loss" in raw.columns)
print("Cleaned    :", clean.shape, "| 'id' present:", "id" in clean.columns)
print("Encoded    :", encoded.shape, "| remaining text columns:",
      len(encoded.select_dtypes(exclude="number").columns))
print("Scaled     :", scaled.shape, "| continuous features scaled:", len(continuous_features))
print("Final      :", final.shape)
print("05 equals 04:", final.equals(scaled))
print("Target 'loss' still unscaled (same values as stage 3):",
      final["loss"].equals(encoded["loss"]))
print("Features   :", X_train.shape[1], "| Training rows:", len(X_train), "| Testing rows:", len(X_test))

Raw        : (50000, 132) | 'id' present: True | 'loss' present: True
Cleaned    : (50000, 131) | 'id' present: False
Encoded    : (50000, 131) | remaining text columns: 0
Scaled     : (50000, 131) | continuous features scaled: 14
Final      : (50000, 131)
05 equals 04: True
Target 'loss' still unscaled (same values as stage 3): True
Features   : 130 | Training rows: 40000 | Testing rows: 10000


## 8. Conclusion

The single authoritative pipeline `Src/claimwise_preprocessing_pipeline.py` was executed and all
nine output files were re-read and verified. Results of this run:

| Check | Actual value |
|---|---|
| Raw dataset | 50,000 x 132 |
| Final preprocessed dataset | 50,000 x 131 (130 input features + `loss`) |
| Train / test split | 40,000 / 10,000 rows, `test_size=0.20`, `random_state=42` |
| Missing values after preprocessing | 0 |
| Duplicate rows after preprocessing | 0 |
| Data-leakage checks | 10 / 10 PASS |
| Scaled-feature mean in `X_train` | 0.000000 (scaler fitted on these rows) |
| Scaled-feature mean in `X_test` | 0.023239 (proves test rows were not used to fit it) |

The stage table is saved to `Outputs/Preprocessing/claimwise_preprocessing_stage_summary.csv`.

Next: `03_ClaimWise_EDA.ipynb`.